# Connect Your App

**What you will build:** calling your deployed agent API (30) from a small front end, so a user can try it directly. Maps to n8n's "Connect Your App" project.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/31_connect_your_app.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

## The shape of it

```
  Browser / app  ──HTTP POST /chat──▶  Your FastAPI service (30)  ──▶  Agent  ──▶  LLM
       ▲                                                                            │
       └──────────────────────  JSON reply (or streamed tokens)  ◀─────────────────┘
```

Your agent is now just an HTTP endpoint. *Any* front end that can make a POST request can use it: a React app, a no-code builder like **Lovable** or **Replit**, a Slack bot, a mobile app. You built the hard part; the front end is a thin client.

## Talk to it: a runnable Gradio client

You don't need to hand-write a UI to test the API. `gradio` gives you a chat box in about ten lines that calls the API from 3.0 and sends the `X-API-Key` header. Point `AGENT_API_URL` at your deployed service (or a local `uvicorn app:app`), set `AGENT_API_KEY` to match the server, and `demo.launch()` opens a shareable chat window — in Colab it even hands you a public link you can open on your phone.

In [ ]:
!pip install -q gradio httpx

import os, httpx, gradio as gr

API_URL = os.getenv("AGENT_API_URL", "http://127.0.0.1:8000/chat")   # your 3.0 service
API_KEY = os.getenv("AGENT_API_KEY", "")                              # must match the server's

async def ask_agent(message, history):
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(API_URL, headers={"X-API-Key": API_KEY},
                              json={"text": message, "session_id": "gradio-demo"})
        r.raise_for_status()
        return r.json()["reply"]

demo = gr.ChatInterface(ask_agent, title="Your Agent")
# demo.launch()        # uncomment to open the chat UI (Colab returns a public share link)

## One thing to add first: CORS

A browser on a different domain than your API will be blocked unless the API allows it. Add CORS to the FastAPI app from 30 (loosely for a demo; restrict `allow_origins` to your real front end in production):

In [ ]:
# Add to app.py (from notebook 30), right after `app = FastAPI()`:
cors_snippet = '''
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # for a demo. In production: your front-end's URL only.
    allow_methods=["*"],
    allow_headers=["*"],
)
'''
print(cors_snippet)

## Calling it from a web page

A minimal front end is a few lines of JavaScript. For the **plain** endpoint, `fetch` and read the JSON:

```html
<input id="q"><button onclick="send()">Ask</button><pre id="out"></pre>
<script>
async function send() {
  const text = document.getElementById('q').value;
  const res = await fetch('http://localhost:8000/chat', {
    method: 'POST',
    headers: {'content-type': 'application/json'},
    body: JSON.stringify({ text })
  });
  const data = await res.json();
  document.getElementById('out').textContent = data.reply;
}
</script>
```

For the **streaming** endpoint, read the response body as a stream and append chunks as they arrive — that's what makes the UI feel alive:

```javascript
const res = await fetch('http://localhost:8000/chat/stream', {
  method: 'POST', headers: {'content-type': 'application/json'},
  body: JSON.stringify({ text })
});
const reader = res.body.getReader(), dec = new TextDecoder();
const out = document.getElementById('out');
while (true) {
  const { value, done } = await reader.read();
  if (done) break;
  out.textContent += dec.decode(value).replaceAll('data: ', '');
}
```

## A cost gate before you share

An API key stops strangers; it does not stop *one* authenticated user (or a runaway script) from hammering the endpoint and running up your bill. A tiny in-memory limiter is enough for a classroom demo — production uses Redis or your API gateway, but the shape is identical: count recent hits per user, refuse past a threshold.

In [ ]:
from time import time

hits = {}   # user_id -> list of recent request timestamps (Redis in production)

def allow(user_id, limit=20, window=3600):
    """True if this user is under `limit` requests in the last `window` seconds."""
    now = time()
    hits[user_id] = [t for t in hits.get(user_id, []) if now - t < window]
    if len(hits[user_id]) >= limit:
        return False
    hits[user_id].append(now)
    return True

# In FastAPI (30) you'd call this per request:
#   if not allow(msg.session_id): raise HTTPException(429, "rate limit exceeded")
print([allow("u-1", limit=3) for _ in range(5)])   # -> True, True, True, False, False

## The no-code shortcut

You don't have to hand-write the front end. Tools like **Lovable**, **Replit**, or **v0** generate a working UI from a prompt — tell them "a chat box that POSTs `{text}` to `<your-url>/chat` and shows `reply`" and you have an app. This is the exact bridge the n8n course used: the agent is the backend, a generated UI is the face.

```{note}
Two production must-dos before you share a URL: **an API key / auth** on your endpoint (an open agent endpoint is an open bill), and **rate limiting**. Never ship a public endpoint that calls a paid model without both.
```

**What's next:** in **32** you build a real **capstone project** — combining tools, memory, guardrails, and evals into one thing worth putting in a portfolio.